# 04 - Survival Analysis & Rule-Based Flags
**Vehicle Predictive Maintenance Project**

---

## Purpose of This Notebook

This notebook produces repair risk signals for five key categories using two different approaches, chosen based on the data available for each category.

### Approach 1 — Survival Analysis (Engine, Electrical, Cooling)
Kaplan-Meier and Weibull AFT models are used for categories where the first occurrence is the meaningful event and the data is sufficient to model it reliably.

### Approach 2 — Rule-Based Flag (Brakes, Wheels & Tyres)
Investigation revealed that the survival model cannot reliably predict brake and tyre replacement cycles for two reasons:
- **Insufficient repeat data** — only 22 vehicles have 2+ brake visits, not enough to learn a replacement cycle
- **Warranty gap** — new vans typically come with a 2-3 year dealer warranty. Repairs during this period are handled by the dealer and never recorded in our system. This makes first-repair durations appear artificially long

Instead, we apply a business rule grounded in domain knowledge: **any van over 3 years old that has not had a recorded brake or tyre repair in the last 24 months is flagged as overdue.**

The 3-year age threshold excludes vehicles still likely under warranty where the absence of a repair record is expected and does not indicate a problem.

## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter, WeibullAFTFitter
from lifelines.statistics import logrank_test
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

PROCESSED = '../data/processed/'
OUTPUTS   = '../outputs/'

maintenance     = pd.read_csv(PROCESSED + 'maintenance_clean.csv', parse_dates=['Invoice Date', 'Booking Date'])
master          = pd.read_csv(PROCESSED + 'master_vehicles.csv', parse_dates=['Registered Date'])
features        = pd.read_csv(PROCESSED + 'features.csv')
features_active = pd.read_csv(PROCESSED + 'features_active.csv')

REFERENCE_DATE  = pd.Timestamp('2025-12-31')
PREDICTION_DAYS = 90

# Categories split by approach
SURVIVAL_CATEGORIES = ['Engine', 'Electrical', 'Cooling']
RULE_CATEGORIES     = ['Brakes', 'Wheels & Tyres']

# Active vehicles only for survival model censoring
master_active = master[master['Vehicle Status'] == 'Current - On Road'].copy()

print('Data loaded.')
print(f'  Maintenance records:  {len(maintenance):,}')
print(f'  Active vehicles:      {len(master_active):,}')
print(f'  Survival model:       {SURVIVAL_CATEGORIES}')
print(f'  Rule-based flag:      {RULE_CATEGORIES}')

## 2. Prepare Repair Timeline

### Why
We use Booking Date as our primary date — this is when the vehicle physically went into the garage. Where Booking Date is missing we fall back to Invoice Date.

We also deduplicate to one row per vehicle per date per category. A single garage visit often generates multiple line items (e.g. front pads, rear pads, discs, labour) all with the same date — we only want to count this as one visit for survival analysis purposes.

In [ ]:
maintenance['Repair_Date'] = maintenance['Booking Date'].fillna(maintenance['Invoice Date'])

print(f'Using Booking Date:            {maintenance["Booking Date"].notnull().sum():,} records')
print(f'Using Invoice Date (fallback):  {maintenance["Booking Date"].isnull().sum():,} records')
print(f'Repair date range: {maintenance["Repair_Date"].min().date()} to {maintenance["Repair_Date"].max().date()}')
print()

# Deduplicated view — one row per vehicle per date per category
maintenance_deduped = maintenance.drop_duplicates(
    subset=['Registration', 'Top Category', 'Repair_Date']
).copy()

print(f'Raw records:        {len(maintenance):,}')
print(f'After dedup:        {len(maintenance_deduped):,}')
print(f'Removed (same-day line items): {len(maintenance) - len(maintenance_deduped):,}')

## 3. Rule-Based Flags — Brakes & Wheels and Tyres

### Why
Rather than using a statistical model that would give unreliable results for these categories, we apply a straightforward business rule:

> **Flag any active van that is over 3 years old and has not had a recorded brake or tyre repair in the last 24 months**

The 3-year age cut-off accounts for dealer warranty periods — vans under 3 years old may have had these repairs done by the dealer without it appearing in our records. Flagging them would produce false positives.

Vehicles over 3 years old with no recent record are genuinely worth reviewing.

In [ ]:
OVERDUE_MONTHS    = 24   # Flag if no repair in last 24 months
WARRANTY_YEARS    = 4    # Exclude vehicles under this age from the flag

overdue_cutoff = REFERENCE_DATE - pd.DateOffset(months=OVERDUE_MONTHS)

# Get active vehicles with age and last repair date per rule category
active_vans = features_active[['Registration', 'Vehicle Age (Years)']].copy()
active_vans['Registered Date'] = pd.to_datetime(
    master_active.set_index('Registration').reindex(active_vans['Registration'])['Registered Date'].values
)

rule_flags = active_vans.copy()

for category in RULE_CATEGORIES:
    col_name = category.replace(' & ', '_').replace(' ', '_')

    # Last repair date per vehicle for this category
    cat_last = (
        maintenance_deduped[maintenance_deduped['Top Category'] == category]
        .groupby('Registration')['Repair_Date']
        .max()
        .reset_index()
        .rename(columns={'Repair_Date': f'Last_{col_name}_Date'})
    )

    rule_flags = rule_flags.merge(cat_last, on='Registration', how='left')

    # Days since last repair in this category
    rule_flags[f'Days_Since_{col_name}'] = (
        REFERENCE_DATE - rule_flags[f'Last_{col_name}_Date']
    ).dt.days

    # Apply flag:
    # Vehicle must be over WARRANTY_YEARS old AND
    # either never had this repair OR last repair was over OVERDUE_MONTHS ago
    over_warranty  = rule_flags['Vehicle Age (Years)'] >= WARRANTY_YEARS
    no_record      = rule_flags[f'Last_{col_name}_Date'].isnull()
    overdue_repair = rule_flags[f'Last_{col_name}_Date'] < overdue_cutoff

    rule_flags[f'{col_name}_Flag'] = (
        over_warranty & (no_record | overdue_repair)
    ).astype(int)

    flagged = rule_flags[f'{col_name}_Flag'].sum()
    eligible = over_warranty.sum()
    print(f'{category}:')
    print(f'  Active vans over {WARRANTY_YEARS} years old: {eligible}')
    print(f'  Flagged as overdue:              {flagged} ({flagged/eligible*100:.1f}%)')
    print()

In [ ]:
# Visualise the flagged vehicles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, category in enumerate(RULE_CATEGORIES):
    col_name = category.replace(' & ', '_').replace(' ', '_')
    flag_col = f'{col_name}_Flag'
    days_col = f'Days_Since_{col_name}'

    # Only show vehicles over warranty age
    eligible = rule_flags[rule_flags['Vehicle Age (Years)'] >= WARRANTY_YEARS].copy()
    eligible[days_col] = np.where(
        eligible[f'Last_{col_name}_Date'].isnull(),
    (REFERENCE_DATE - eligible['Registered Date']).dt.days,  # Days since registration
    (REFERENCE_DATE - eligible[f'Last_{col_name}_Date']).dt.days  # Days since last repair  # Never repaired = very overdue
    )

    colors = ['coral' if f == 1 else 'steelblue' for f in eligible[flag_col]]
    axes[i].scatter(
        eligible['Vehicle Age (Years)'],
        eligible[days_col],
        c=colors, alpha=0.6, s=20
    )
    axes[i].axhline(OVERDUE_MONTHS * 30, color='red', linestyle='--',
                    label=f'{OVERDUE_MONTHS}m threshold')
    axes[i].axvline(WARRANTY_YEARS, color='orange', linestyle='--',
                    label=f'{WARRANTY_YEARS}yr warranty cutoff')
    axes[i].set_title(f'{category} — Overdue Flag')
    axes[i].set_xlabel('Vehicle Age (Years)')
    axes[i].set_ylabel('Days Since Last Repair')
    axes[i].legend(fontsize=8)

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='coral', label='Flagged'),
        Patch(facecolor='steelblue', label='OK')
    ]
    axes[i].legend(handles=legend_elements + axes[i].get_lines(), fontsize=8)

plt.suptitle('Rule-Based Overdue Flags — Brakes & Wheels/Tyres', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/04_rule_based_flags.png', dpi=150)
plt.show()

## 4. Survival Analysis — Engine, Electrical, Cooling

### Why
For these three categories the survival model is appropriate:
- The first occurrence of the repair is meaningful — an engine fault on a relatively new van is a genuine event worth capturing
- The data has sufficient events to produce reliable curves
- There is no warranty distortion issue — dealer warranty is unlikely to cover all electrical and cooling repairs

We build the survival dataset using active vehicles only for censored observations, deduplicated to one visit per vehicle per date.

In [ ]:
def build_survival_dataset(category, maintenance_df, master_df):
    """
    Build a time-based survival dataset for a given repair category.
    Uses deduplicated maintenance records (one visit per vehicle per date).
    For vehicles WITH repairs: duration = days from registration or previous visit
    For vehicles WITHOUT repairs: duration = days from registration to reference date (censored)
    """
    cat_repairs = maintenance_df[maintenance_df['Top Category'] == category].copy()
    cat_repairs = cat_repairs.drop_duplicates(subset=['Registration', 'Repair_Date'])
    cat_repairs = cat_repairs.sort_values(['Registration', 'Repair_Date'])
    records = []

    for reg, group in cat_repairs.groupby('Registration'):
        group = group.sort_values('Repair_Date').reset_index(drop=True)
        vehicle_row = master_df[master_df['Registration'] == reg]
        reg_date = pd.Timestamp(vehicle_row['Registered Date'].values[0]) if not vehicle_row.empty else None

        for i in range(len(group)):
            start = reg_date if (i == 0 and reg_date is not None) else (
                group.loc[i-1, 'Repair_Date'] if i > 0 else group.loc[0, 'Repair_Date']
            )
            duration = (group.loc[i, 'Repair_Date'] - start).days
            if duration > 0:
                records.append({'Registration': reg, 'duration': duration, 'event_observed': 1})

    # Censored — active vehicles with no repair in this category
    repaired_vans = set(cat_repairs['Registration'])
    for reg in set(master_df['Registration']) - repaired_vans:
        vehicle_row = master_df[master_df['Registration'] == reg]
        if vehicle_row.empty:
            continue
        reg_date = vehicle_row['Registered Date'].values[0]
        if pd.isna(reg_date):
            continue
        duration = (REFERENCE_DATE - pd.Timestamp(reg_date)).days
        if duration > 0:
            records.append({'Registration': reg, 'duration': duration, 'event_observed': 0})

    df = pd.DataFrame(records)
    return df[df['duration'] > 0].dropna(subset=['duration'])

print('Survival dataset builder defined.')

In [ ]:
survival_datasets = {}

for category in SURVIVAL_CATEGORIES:
    df = build_survival_dataset(category, maintenance_deduped, master_active)
    survival_datasets[category] = df
    events   = int(df['event_observed'].sum())
    censored = int((df['event_observed'] == 0).sum())
    med_dur  = df[df['event_observed'] == 1]['duration'].median()
    print(f'{category:15s} — {len(df):>4} records | {events:>3} events | {censored:>4} censored | median interval: {med_dur:.0f} days')

## 5. Kaplan-Meier Survival Curves

### Why
The Kaplan-Meier estimator shows the probability a vehicle has NOT yet needed this repair at each point in time. The median survival time is the number of days at which 50% of vehicles have needed that repair — a useful fleet-level benchmark.

In [ ]:
kmf_models = {}
fig, ax = plt.subplots(figsize=(14, 6))

for category in SURVIVAL_CATEGORIES:
    df  = survival_datasets[category]
    kmf = KaplanMeierFitter()
    kmf.fit(durations=df['duration'], event_observed=df['event_observed'], label=category)
    kmf_models[category] = kmf
    kmf.plot_survival_function(ax=ax, ci_show=True)

ax.set_title('Kaplan-Meier Survival Curves — Engine, Electrical, Cooling', fontsize=13)
ax.set_xlabel('Days Since Registration or Previous Repair')
ax.set_ylabel('Probability of No Repair Needed')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/04_km_curves.png', dpi=150)
plt.show()

print('Median survival per category:')
for category, kmf in kmf_models.items():
    median = kmf.median_survival_time_
    if np.isinf(median):
        print(f'  {category:15s} — median not reached (>50% vehicles yet to have this repair)')
    else:
        print(f'  {category:15s} — {median:>6,.0f} days  (~{median/30.4:.1f} months)')

In [ ]:
# Individual curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, category in enumerate(SURVIVAL_CATEGORIES):
    kmf = kmf_models[category]
    kmf.plot_survival_function(ax=axes[i], ci_show=True)
    axes[i].set_title(category)
    axes[i].set_xlabel('Days')
    axes[i].set_ylabel('Survival Probability')
    axes[i].get_legend().remove()
    median = kmf.median_survival_time_
    if not np.isnan(median) and not np.isinf(median):
        axes[i].axvline(median, color='red', linestyle='--', alpha=0.7, label=f'Median: {median:.0f}d')
        axes[i].legend(fontsize=8)

plt.suptitle('Survival Curves — Engine, Electrical, Cooling (95% CI)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS + 'reports/04_km_curves_individual.png', dpi=150)
plt.show()

## 6. Statistical Comparison Between Categories

In [ ]:
print('Log-rank test (p < 0.05 = significantly different repair timing):\n')
for cat_a, cat_b in combinations(SURVIVAL_CATEGORIES, 2):
    df_a   = survival_datasets[cat_a]
    df_b   = survival_datasets[cat_b]
    result = logrank_test(
        df_a['duration'], df_b['duration'],
        event_observed_A=df_a['event_observed'],
        event_observed_B=df_b['event_observed']
    )
    sig = '*** SIGNIFICANT' if result.p_value < 0.05 else ''
    print(f'  {cat_a:15s} vs {cat_b:15s} — p={result.p_value:.4f} {sig}')

## 7. Weibull AFT Model — Vehicle Feature Influence

### Why
The Weibull AFT model incorporates vehicle features to explain why some vehicles need repairs sooner. A negative coefficient means that feature accelerates time to repair. A positive coefficient extends time between repairs.

This tells us whether factors like driver behaviour, daily mileage, and vehicle age are genuinely predictive for each repair category.

In [ ]:
AFT_FEATURES   = ['Registration', 'Vehicle Age (Years)', 'DriverScore',
                  'SpeedingRate', 'DeccelerationRate', 'Avg_Daily_Miles_Recent', 'Asset Type_Encoded']
aft_feature_df = features[AFT_FEATURES].copy()

def fit_aft_model(category, survival_datasets, aft_feature_df):
    df = survival_datasets[category].copy()
    df = df.merge(aft_feature_df, on='Registration', how='left')
    model_cols = ['duration', 'event_observed', 'Vehicle Age (Years)', 'DriverScore',
                  'SpeedingRate', 'DeccelerationRate', 'Avg_Daily_Miles_Recent', 'Asset Type_Encoded']
    df = df[model_cols].dropna()
    if len(df) < 20:
        print(f'  {category}: insufficient data ({len(df)} records)')
        return None
    aft = WeibullAFTFitter()
    aft.fit(df, duration_col='duration', event_col='event_observed')
    return aft

aft_models = {}
for category in SURVIVAL_CATEGORIES:
    print(f'Fitting AFT model: {category}')
    model = fit_aft_model(category, survival_datasets, aft_feature_df)
    if model is not None:
        aft_models[category] = model
        print(f'  Fitted on {len(survival_datasets[category])} records')

In [ ]:
for category, model in aft_models.items():
    print(f'\n{"="*55}')
    print(f'AFT Model — {category}')
    print(f'{"="*55}')
    print(model.summary[['coef', 'exp(coef)', 'p']].round(4))
    print('exp(coef) > 1 = extends time to repair | < 1 = accelerates')

In [ ]:
n_models = len(aft_models)
if n_models > 0:
    fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
    if n_models == 1:
        axes = [axes]
    for ax, (category, model) in zip(axes, aft_models.items()):
        summary = model.summary
        params  = summary[summary.index.get_level_values(0) == 'lambda_'].copy()
        params.index = params.index.get_level_values(1)
        params  = params[params.index != 'Intercept']
        colors  = ['coral' if v < 0 else 'steelblue' for v in params['coef']]
        params['coef'].plot(kind='barh', ax=ax, color=colors)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(category)
        ax.set_xlabel('Coefficient (negative = faster repair)')
    plt.suptitle('AFT Feature Coefficients', fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUTS + 'reports/04_aft_coefficients.png', dpi=150)
    plt.show()

## 8. Per-Vehicle Survival Predictions (Engine, Electrical, Cooling)

### Why
For each active vehicle we calculate the probability of needing each survival-modelled repair within the next 90 days, using the KM curve evaluated at the 90-day horizon.

In [ ]:
survival_predictions = features_active[[
    'Registration', 'Vehicle Status', 'Vehicle Age (Years)',
    'Cumulative Miles', 'Days_Since_Last_Repair'
]].copy()

for category in SURVIVAL_CATEGORIES:
    col_name = category.replace(' & ', '_').replace(' / ', '_').replace(' ', '_')
    if category not in kmf_models:
        continue
    kmf = kmf_models[category]
    sf  = kmf.survival_function_
    t_horizon       = min(PREDICTION_DAYS, sf.index.max())
    prob_repair_90d = round(1 - float(kmf.predict(t_horizon)), 4)
    median_days     = kmf.median_survival_time_
    median_days     = None if np.isinf(median_days) else round(float(median_days), 0)

    survival_predictions[f'Prob_{col_name}_90d']    = prob_repair_90d
    survival_predictions[f'Median_Days_{col_name}'] = median_days

# Join rule-based flags
rule_flag_cols = ['Registration'] + [c for c in rule_flags.columns if c.endswith('_Flag') or c.startswith('Days_Since_')]
survival_predictions = survival_predictions.merge(
    rule_flags[rule_flag_cols], on='Registration', how='left'
)

print(f'Predictions generated for {len(survival_predictions):,} active vehicles.')
print(f'\nFleet-average repair probability within {PREDICTION_DAYS} days:')
for col in [c for c in survival_predictions.columns if c.startswith('Prob_')]:
    print(f'  {col:45s} {survival_predictions[col].mean():.1%}')

print(f'\nRule-based flags (vehicles over {WARRANTY_YEARS} years old):')
for col in [c for c in survival_predictions.columns if c.endswith('_Flag')]:
    print(f'  {col:35s} {int(survival_predictions[col].sum())} vehicles flagged')

In [ ]:
both_flagged = survival_predictions[
    (survival_predictions['Brakes_Flag'] == 1) & 
    (survival_predictions['Wheels_Tyres_Flag'] == 1)
]
print(f'Vehicles flagged for BOTH brakes and tyres: {len(both_flagged)}')

## 9. Save Outputs

In [ ]:
survival_predictions.to_csv(PROCESSED + 'survival_predictions.csv', index=False)

km_summary = []
for category, kmf in kmf_models.items():
    df     = survival_datasets[category]
    median = kmf.median_survival_time_
    km_summary.append({
        'Category':         category,
        'Approach':         'Survival Model',
        'Total Records':    len(df),
        'Events (Repairs)': int(df['event_observed'].sum()),
        'Censored':         int((df['event_observed'] == 0).sum()),
        'Median Days':      None if np.isinf(median) else round(float(median), 0),
        'Median Months':    None if np.isinf(median) else round(float(median) / 30.4, 1)
    })

for category in RULE_CATEGORIES:
    col_name = category.replace(' & ', '_').replace(' ', '_')
    flagged  = int(survival_predictions[f'{col_name}_Flag'].sum()) if f'{col_name}_Flag' in survival_predictions.columns else 0
    km_summary.append({
        'Category':         category,
        'Approach':         f'Rule-Based (>{WARRANTY_YEARS}yr, no repair in {OVERDUE_MONTHS}m)',
        'Total Records':    None,
        'Events (Repairs)': None,
        'Censored':         None,
        'Median Days':      None,
        'Median Months':    None
    })

km_summary_df = pd.DataFrame(km_summary)
km_summary_df.to_csv(PROCESSED + 'km_summary.csv', index=False)

print('Outputs saved:')
print(f'  survival_predictions.csv — {len(survival_predictions):,} active vehicles')
print(f'  km_summary.csv')
print()
display(km_summary_df)

## 10. Summary

In [ ]:
print(f"""
========================================================
 SURVIVAL ANALYSIS & RULE-BASED FLAGS SUMMARY
========================================================

SURVIVAL MODEL (Engine, Electrical, Cooling):
  - Time-based duration (days between repairs)
  - Booking Date primary, Invoice Date fallback
  - Deduplicated to one visit per vehicle per date
  - Active vehicles only used for censored observations
  - Kaplan-Meier + Weibull AFT models fitted
  - Prediction horizon: {PREDICTION_DAYS} days

RULE-BASED FLAG (Brakes, Wheels & Tyres):
  - Survival model not suitable — insufficient repeat data
    and warranty gap distorts first-repair durations
  - Flag: vehicle age > {WARRANTY_YEARS} years AND no repair in last {OVERDUE_MONTHS} months
  - Vehicles under {WARRANTY_YEARS} years excluded (likely under dealer warranty)

KNOWN LIMITATIONS:
  - Dealer warranty repairs (first ~3 years) not captured
  - This affects all categories to some degree
  - Repair data starts mid-fleet-lifecycle for many vehicles

OUTPUTS:
  - survival_predictions.csv (survival probs + rule flags)
  - km_summary.csv
  - 04_km_curves.png
  - 04_km_curves_individual.png
  - 04_aft_coefficients.png
  - 04_rule_based_flags.png

NEXT: Notebook 05 — Risk Classification
========================================================
""")